# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q emoji
    print('Packages installed.')

    # Download NLTK data
    import nltk
    nltk.download('punkt', quiet=True)
    nltk.download('stopwords', quiet=True)
    nltk.download('wordnet', quiet=True)
    nltk.download('omw-1.4', quiet=True)
    nltk.download('punkt_tab', quiet=True)
    print('NLTK data downloaded.')


# Text Preprocessing Pipeline In-Depth

Preprocessing is the foundation of NLP. A bad cleaning pipeline creates "garbage in, garbage out". This notebook demonstrates a robust pipeline handling URLs, HTML tags, emojis, and compares Stemming vs. Lemmatization.

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
import emoji

# Ensure resources are downloaded
try:
    nltk.data.find('corpora/stopwords')
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('stopwords', quiet=True)
    nltk.download('wordnet', quiet=True)
    nltk.download('omw-1.4', quiet=True)


## 1. The Messy Text
Real-world text, especially from the web, is messy.

In [ ]:
text = """
Check out this link: https://example.com! 
I am SO happy today 😀!!! 
Running, runs, and ran are all forms of run. <br> 
The <b>quick</b> brown foxes were jumping over the lazy dogs.
"""
print("Original:")
print(text)

## 2. Cleaning Steps
We define functions for each step.

In [ ]:
def remove_urls(text):
    return re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

def remove_html(text):
    return re.sub(r'<.*?>', '', text)

def convert_emojis(text):
    return emoji.demojize(text) # Converts 😀 to :grinning_face:

def remove_special_chars(text):
    # Remove everything that isn't a letter or space
    return re.sub(r'[^a-zA-Z\s]', '', text)

cleaned = remove_urls(text)
cleaned = remove_html(cleaned)
cleaned = convert_emojis(cleaned)
cleaned = remove_special_chars(cleaned)
cleaned = cleaned.lower()

print("Cleaned:")
print(cleaned)

## 3. Stemming vs. Lemmatization
*   **Stemming**: Chops off ends of words (heuristic). Fast but can produce non-words (e.g., "ponies" -> "poni").
*   **Lemmatization**: Uses a dictionary to find the root form (lemma). Slower but accurate (e.g., "ponies" -> "pony").

In [ ]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

words = cleaned.split()
stop_words = set(stopwords.words('english'))
words = [w for w in words if w not in stop_words]

print(f"{'Word':<15} {'Stem':<15} {'Lemma':<15}")
print("-" * 45)

for w in words:
    print(f"{w:<15} {stemmer.stem(w):<15} {lemmatizer.lemmatize(w):<15}")

## 4. Reusable Pipeline
Wrap it all in a function for your projects.

In [ ]:
def clean_text_pipeline(text):
    text = remove_urls(text)
    text = remove_html(text)
    text = emoji.replace_emoji(text, replace='') # Remove emojis entirely if preferred
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.lower()
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words]
    return ' '.join(tokens)

print("Final Output:")
print(clean_text_pipeline(text))